### Задание 1 (наследование и перегрузка операторов)

In [52]:
class Str(str):
    def __lt__(self, other):
        if len(self) != len(other):
            return len(self) < len(other)
        return super().__lt__(other)

    def __le__(self, other):
        if len(self) != len(other):
            return len(self) <= len(other)
        return super().__le__(other)

    def __gt__(self, other):
        if len(self) != len(other):
            return len(self) > len(other)
        return super().__gt__(other)

    def __ge__(self, other):
        if len(self) != len(other):
            return len(self) >= len(other)
        return super().__ge__(other)

    def __eq__(self, other):
        return super().__eq__(other)

    def __ne__(self, other):
        return super().__ne__(other)

In [53]:
Str('ac') < Str('a')

False

In [54]:
Str('ac') > Str('a')

True

In [55]:
Str('ac') < Str('aba')

True

In [56]:
Str('ac') > Str('aba')

False

In [57]:
Str('ac').upper()

'AC'

### Задание 2 (полиморфизм и инкапсуляция)


In [71]:
def calculate_perimeter(figure):
    return figure.get_perimeter()

def calculate_square(figure):
    return figure.get_square()

In [72]:
import math

class Rectangle:
    def __init__(self, a, b):
        self.a = a
        self.b = b

    def get_perimeter(self):
        return 2 * (self.a + self.b)

    def get_square(self):
        return self.a * self.b


class Circle:
    def __init__(self, r):
        self.r = r

    def get_perimeter(self):
        return 2 * math.pi * self.r

    def get_square(self):
        return math.pi * self.r ** 2


class Rhombus:
    def __init__(self, p, q):
        self.p = p
        self.q = q

    def get_perimeter(self):
        side = math.sqrt((self.p / 2) ** 2 + (self.q / 2) ** 2)
        return 4 * side

    def get_square(self):
        return (self.p * self.q) / 2

In [59]:
rec = Rectangle(2, 3)
rec.get_perimeter(), rec.get_square()

(10, 6)

In [60]:
cir = Circle(3)
cir.get_perimeter(), cir.get_square()

(18.84955592153876, 28.274333882308138)

In [61]:
rhom = Rhombus(3, 4)
rhom.get_perimeter(), rhom.get_square()

(10.0, 6.0)

In [73]:
calculate_perimeter(cir)

18.84955592153876

### Задание 3 (наследование)

In [69]:
from collections import Counter

class MyCounter(Counter):
    def least_common(self, n=None):
        if n is None:
            return sorted(self.items(), key=lambda x: x[1])
        return sorted(self.items(), key=lambda x: x[1])[:n]

In [70]:
a = 'strings'
print(MyCounter(a).least_common(2))
print(MyCounter(a).most_common(2))

[('t', 1), ('r', 1)]
[('s', 2), ('t', 1)]


### Задание 4 (абстракция)

In [64]:
from functools import wraps

def unexcept(fun):
    """
    Этот декоратор не позволяет функции возражать исключения
    """
    @wraps(fun)
    def wrapped(*args, **kwargs):
        e = None
        try:
            res = fun(*args, **kwargs)
        except Exception as ex:
            res, e = 0, ex
        return res, e

    return wrapped

In [66]:
class Deposit:
    def __init__(self, percent, amount, period,
                 capitalization=True, replenishment=False,
                 partial_withdrawal=False, auto_prolongation=False):
        self.percent = percent / 100
        self.amount = amount
        self.period = period
        self.capitalization = capitalization
        self.replenishment = replenishment
        self.partial_withdrawal = partial_withdrawal
        self.auto_prolongation = auto_prolongation
        self.opened = True
        self.start_month = 0

    def calculate_interest(self, months):
        """Рассчитать сумму через months месяцев без изменения вклада"""
        if self.capitalization:
            return self.amount * (1 + self.percent / 12) ** months
        else:
            return self.amount * (1 + self.percent / 12 * months)

    @unexcept
    def wait(self, months):
        """Подождать months месяцев, начислить проценты"""
        if not self.opened:
            raise Exception("Депозит закрыт")
        
        if self.capitalization:
            for _ in range(months):
                monthly_interest = self.amount * (self.percent / 12)
                self.amount += monthly_interest
        else:
            total_interest = self.amount * (self.percent / 12) * months
        
        self.start_month += months
        if self.start_month >= self.period:
            if self.auto_prolongation:
                self.start_month = 0
            else:
                self.opened = False

    @unexcept
    def replenish(self, amount):
        if not self.replenishment:
            raise Exception("Пополнение запрещено")
        if not self.opened:
            raise Exception("Депозит закрыт")
        self.amount += amount

    @unexcept
    def withdraw(self, amount):
        if not self.partial_withdrawal:
            raise Exception("Частичное снятие запрещено")
        if not self.opened:
            raise Exception("Депозит закрыт")
        if amount > self.amount:
            raise Exception("Недостаточно средств")
        self.amount -= amount

    def close(self):
        self.opened = False
        return self.amount

In [67]:
deposit1 = Deposit(15, 10000, 12, replenishment = True, auto_prolongation = True, partial_withdrawal = True)
print(deposit1.calculate_interest(0))
deposit1.replenish(10000)
print(deposit1.calculate_interest(0))
deposit1.withdraw(5000)
print(deposit1.calculate_interest(0))
deposit1.wait(12)
print(deposit1.calculate_interest(0))
deposit1.withdraw(10000)
print(deposit1.withdraw(1000000))
deposit1.close()
print(deposit1.replenish(50000))

deposit2 = Deposit(20, 1000, 24)
print(deposit2.withdraw(10))
print(deposit2.replenish(10))

10000.0
20000.0
15000.0
17411.317765844982
(0, Exception('Недостаточно средств'))
(0, Exception('Депозит закрыт'))
(0, Exception('Частичное снятие запрещено'))
(0, Exception('Пополнение запрещено'))
